In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

def clean_sales_file(in_path, out_dir):
    
    skip = get_skiprows_from_filename(in_path)
    
    df = pd.read_excel(in_path, skiprows=skip).dropna(how="all").reset_index(drop=True)

    #Clean Column Names

    df.columns=(
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\n]+", "_", regex=True)) #Replaces spaces and newlines in column names with underscores 


    borough_map = {'1': 'Manhattan', '2': 'Bronx', '3': 'Brooklyn', '4': 'Queens', '5': 'Staten Island'}
    if 'borough' in df.columns:
        df['borough'] = (
            df['borough']
            .astype(str)
            .str.strip()
            .str[0]             
            .map(borough_map)
        )
        
    #Clean sale_price
    df['sale_price']=(
        df['sale_price']
        .astype(str)
        .str.replace(r'[\$,]', '', regex=True)
        .replace({'': None, 'nan': None})
        .astype(float)
    )

    # Fill missing residential_units with 1 (since all missing ones are single units)
    if 'total_units' in df.columns:
        df['total_units'] = df['total_units'].fillna(1).astype(int)

    #Sale price per unit for sales of multiple units
    df['price_per_unit'] = np.where(
        (df['total_units'] > 0) & (df['total_units'].notna()),
        df['sale_price'] / df['total_units'],
        np.nan
    )


    #Clean sale_date by converting to datetime
    df['sale_date'] = pd.to_datetime(df['sale_date'], errors='coerce') 

    #Filter invalid rows
    df = df[df['sale_price'].notna() & df['sale_date'].notna()] #Drop any row with missing value sale_price or sale_date

    df = df[(df['sale_price'] >= 10_000) & (df['sale_price'] <= 100_000_000)] #Drop any non-real sales or outliers

    #Extract time information
    df['sale_year']  = df['sale_date'].dt.year
    df['sale_month'] = df['sale_date'].dt.to_period('M').astype(str) 

    #Clean numeric columns
    for col in ['residential_units','commercial_units',
                'total_units','year_built','block','lot']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce') #Ensures numeric columns contain numeric values       

    if 'zip_code' in df.columns:
        df['zip_code'] = (
            df['zip_code']
            .astype(str)
            .str.extract(r'(\d{3,5})', expand=False)  # keep digits
            .str.zfill(5))
        
        

    keep_cols = [
        "borough", "neighborhood", "building_class_category",
        "address", "zip_code", "residential_units", "commercial_units", "total_units", 
        "year_built",
        "sale_price", "sale_date", "sale_year", "sale_month"
    ]
    df = df[[col for col in keep_cols if col in df.columns]]

    
    out_path = out_dir / f"{in_path.stem}_clean.csv"
    df.to_csv(out_path, index=False)
    return out_path


In [2]:
def get_skiprows_from_filename(file_path):
    # Extract a 4-digit year from the filename
    year = re.search(r"(20\d{2})", file_path.stem)
    year = int(year.group(1))

    if 2020 <= year <= 2025:
        return 6   # header starts at row 7
    elif 2011 <= year <= 2019:
        return 4   # header starts at row 5
    elif 2003 <= year <= 2010:
        return 3   # header starts at row 4
    else:
        return 6   # fallback

In [3]:
from pathlib import Path

years = range(2014, 2025)

for yr in years:
    raw_dir = Path(f"../data_raw/{yr}")
    out_dir = Path(f"../data_cleaned/{yr}")
    

    # Collect both .xls and .xlsx files
    excel_files = list(raw_dir.glob("*.xlsx")) + list(raw_dir.glob("*.xls"))
    

    # Clean each file
    for file_path in excel_files:
        cleaned_path = clean_sales_file(file_path, out_dir)
        print(f"✅ {yr}: cleaned and saved {cleaned_path.name}")


✅ 2014: cleaned and saved 2014_statenisland_clean.csv
✅ 2014: cleaned and saved 2014_queens_clean.csv
✅ 2014: cleaned and saved 2014_manhattan_clean.csv
✅ 2014: cleaned and saved 2014_brooklyn_clean.csv
✅ 2014: cleaned and saved 2014_bronx_clean.csv
✅ 2015: cleaned and saved 2015_statenisland_clean.csv
✅ 2015: cleaned and saved 2015_queens_clean.csv
✅ 2015: cleaned and saved 2015_manhattan_clean.csv
✅ 2015: cleaned and saved 2015_brooklyn_clean.csv
✅ 2015: cleaned and saved 2015_bronx_clean.csv
✅ 2016: cleaned and saved 2016_statenisland_clean.csv
✅ 2016: cleaned and saved 2016_queens_clean.csv
✅ 2016: cleaned and saved 2016_manhattan_clean.csv
✅ 2016: cleaned and saved 2016_bronx_clean.csv
✅ 2016: cleaned and saved 2016_brooklyn_clean.csv
✅ 2017: cleaned and saved 2017_statenisland_clean.csv
✅ 2017: cleaned and saved 2017_queens_clean.csv
✅ 2017: cleaned and saved 2017_manhattan_clean.csv
✅ 2017: cleaned and saved 2017_brooklyn_clean.csv
✅ 2017: cleaned and saved 2017_bronx_clean.csv


In [4]:
from pathlib import Path
import pandas as pd

def load_all_cleaned(start=2003, end=2025):
    frames = []
    for yr in range(start, end+1):
        folder = Path(f"../data_cleaned/{yr}")
        for f in folder.glob("*_clean.csv"):
            df = pd.read_csv(f, parse_dates=["sale_date"])
            frames.append(df)
    return pd.concat(frames, ignore_index=True)


# 1) Build
all_sales = load_all_cleaned(2003, 2025)
print(f"Total rows: {len(all_sales):,}")
print("Columns:", list(all_sales.columns))

# 2) Quick sanity checks
print("\nRows by year:")
print(all_sales["sale_year"].value_counts().sort_index().to_string())

print("\nRows by borough (non-null):")
print(all_sales["borough"].dropna().value_counts().to_string())

# 3) Save master dataset
warehouse = Path("../warehouse")
warehouse.mkdir(parents=True, exist_ok=True)

parquet_path = warehouse / "nyc_sales_all.parquet"

all_sales.to_parquet(parquet_path, index=False)

Total rows: 637,536
Columns: ['borough', 'neighborhood', 'building_class_category', 'address', 'zip_code', 'residential_units', 'commercial_units', 'total_units', 'year_built', 'sale_price', 'sale_date', 'sale_year', 'sale_month']

Rows by year:
2014    59850
2015    62907
2016    60755
2017    60892
2018    56842
2019    55937
2020    43792
2021    70813
2022    65714
2023    49469
2024    50565

Rows by borough (non-null):
Queens           189690
Manhattan        167433
Brooklyn         165961
Staten Island     61877
Bronx             52575


In [5]:
df = pd.read_parquet("../warehouse/nyc_sales_all.parquet")

df.to_csv("../warehouse/nyc_sales_all_tableau.csv", index=False)

print(df.head())

         borough               neighborhood  \
0  Staten Island  ANNADALE                    
1  Staten Island  ANNADALE                    
2  Staten Island  ANNADALE                    
3  Staten Island  ANNADALE                    
4  Staten Island  ANNADALE                    

                        building_class_category  \
0  01  ONE FAMILY DWELLINGS                       
1  01  ONE FAMILY DWELLINGS                       
2  01  ONE FAMILY DWELLINGS                       
3  01  ONE FAMILY DWELLINGS                       
4  01  ONE FAMILY DWELLINGS                       

                                     address  zip_code  residential_units  \
0  4716 AMBOY ROAD                             10312.0                1.0   
1  381 HAROLD AVENUE                           10312.0                1.0   
2  24 ELMBANK STREET                           10312.0                1.0   
3  39 SANDGAP STREET                           10312.0                1.0   
4  25 OCEAN DRIVEWAY     